# Ledger Lens — Data Understanding & Exploratory Analysis
## Problem Statement: SIH26102 (MPLADS AI Risk Intelligence & Monitoring)
**Phase 1: Dataset Inspection, Schema Analysis & Quality Audit**

### Objective
This notebook performs a comprehensive audit of the raw data available in `data/raw/` to understand its schema, data quality, distributions, and readiness for risk intelligence modeling.

### Core Principle
> **"AI flags the risk. Evidence explains it. Humans make the decision."**  
> We never fabricate columns, labels, or fraud verdicts. All analytical boundaries are grounded strictly in real data.


In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Ensure UTF-8 output formatting if supported
if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

# Display settings
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Environment initialized successfully.")


Environment initialized successfully.


## 1. Raw Dataset Discovery
Inspect all files currently placed in the `data/raw/` directory.


In [2]:
raw_data_dir = os.path.join("..", "data", "raw")
if not os.path.exists(raw_data_dir):
    raw_data_dir = os.path.join("data", "raw")

files = os.listdir(raw_data_dir)
print(f"Discovered {len(files)} file(s) in {raw_data_dir}:")
for f in files:
    fpath = os.path.join(raw_data_dir, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f" - Filename: {f} | Size: {size_kb:.2f} KB | Type: {f.split('.')[-1].upper()}")


Discovered 1 file(s) in data\raw:
 - Filename: Allocated Limit for Honble MPs.csv | Size: 35.30 KB | Type: CSV


## 2. Dataset Loading & Structural Inspection
Load the official dataset `Allocated Limit for Honble MPs.csv` and inspect its initial shape, column names, and raw data types.


In [3]:
csv_file = os.path.join(raw_data_dir, "Allocated Limit for Honble MPs.csv")
raw_df = pd.read_csv(csv_file)

print(f"Dataset Shape: {raw_df.shape[0]} rows × {raw_df.shape[1]} columns\n")
print("Column Names:")
for i, col in enumerate(raw_df.columns):
    print(f"  [{i}] '{col}' (dtype: {raw_df[col].dtype})")


Dataset Shape: 544 rows × 5 columns

Column Names:
  [0] 'Sr. No.' (dtype: str)
  [1] 'State' (dtype: str)
  [2] 'Hon'ble Members of Parliaments' (dtype: str)
  [3] 'Constituency' (dtype: str)
  [4] 'Allocated AMOUNT ( ₹ )' (dtype: str)


## 3. First & Last Rows Inspection
Let's inspect the first 5 records and the last 5 records to understand the record layout and uncover any header/trailer anomalies.


In [4]:
print("--- First 5 Rows ---")
print(raw_df.head(5).to_string())

print("\n--- Last 5 Rows ---")
print(raw_df.tail(5).to_string())


--- First 5 Rows ---
  Sr. No.              State  Hon'ble Members of Parliaments   Constituency Allocated AMOUNT ( ₹ )
0       1        Maharashtra  AASHTIKAR PATIL NAGESH BAPURAO        HINGOLI              190289442
1       2  Jammu And Kashmir             ABDUL RASHID SHEIKH     BARAMULLAH           154773472.11
2       3              Bihar               ABHAY KUMAR SINHA  AURANGABAD_BR              147000000
3       4        West Bengal            ABHIJIT GANGOPADHYAY         TAMLUK              147000000
4       5        West Bengal                  Abu Taher Khan    MURSHIDABAD              147000000

--- Last 5 Rows ---
         Sr. No.           State           Hon'ble Members of Parliaments          Constituency Allocated AMOUNT ( ₹ )
539          540  Andhra Pradesh                        Y S Avinash Reddy                KADAPA           156430260.11
540          541       Karnataka  YADUVEER KRISHNADATTA CHAMARAJA WADIYAR                MYSORE              147000000
541    

## 4. Missing Values & Duplicate Records Analysis
Calculate missing values per column (count and percentage) and inspect duplicate rows.


In [5]:
# Missing Values
null_counts = raw_df.isnull().sum()
null_pct = (null_counts / len(raw_df)) * 100
missing_report = pd.DataFrame({
    'Column Name': raw_df.columns,
    'Missing Count': null_counts.values,
    'Missing Percentage (%)': null_pct.values
})
print("Missing Values Summary:")
print(missing_report.to_string(index=False))

# Duplicate Rows
exact_duplicates = raw_df.duplicated().sum()
print(f"\nExact Duplicate Rows Count: {exact_duplicates}")


Missing Values Summary:
                   Column Name  Missing Count  Missing Percentage (%)
                       Sr. No.              0                    0.00
                         State              0                    0.00
Hon'ble Members of Parliaments              0                    0.00
                  Constituency              0                    0.00
        Allocated AMOUNT ( ₹ )              1                    0.18

Exact Duplicate Rows Count: 0


## 5. Data Quality Audit & Hygiene Issues
Key observations from raw inspection:
1. **Trailer Summary Row**: Row index `543` is a "Grand Total" summary row (`Sr. No. = 'Grand Total'`, `State = ' '`, `Hon'ble Members of Parliaments = ' '`, `Constituency = ' '`, and aggregate sum `83,33,99,05,622.01`). If not separated, it would corrupt all statistical aggregations.
2. **Column Name Encoding**: Column 4 contains the Indian Rupee symbol: `'Allocated AMOUNT ( ₹ )'`.
3. **Number Formatting**: The amount column contains string formatting with commas (`83,33,99,05,622.01`) and floating decimals.
4. **Text Normalization**: Constituency names contain reservation qualifiers like `(SC)`, `(ST)` and state tags `_BR`, `_UP`. MP names have inconsistent uppercase vs title-case formatting.


In [6]:
# Separate data rows from summary trailer
is_grand_total = raw_df['Sr. No.'].astype(str).str.strip().str.lower() == 'grand total'
df_cleaned = raw_df[~is_grand_total].copy()
grand_total_row = raw_df[is_grand_total].copy()

# Clean amount column: strip whitespace, remove commas, convert to float
amount_col = [c for c in df_cleaned.columns if 'AMOUNT' in c.upper()][0]
df_cleaned['allocated_amount_clean'] = (
    df_cleaned[amount_col]
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
    .astype(float)
)

print(f"Cleaned individual MP records count: {len(df_cleaned)}")
print(f"Grand total row amount: {grand_total_row[amount_col].values[0]}")
print(f"Calculated sum of cleaned MP amounts: ₹{df_cleaned['allocated_amount_clean'].sum():,.2f}")
verification_match = np.isclose(df_cleaned['allocated_amount_clean'].sum(), 83339905622.01)
print(f"Integrity Check - Cleaned Sum Matches Official Grand Total: {verification_match}")


Cleaned individual MP records count: 543
Grand total row amount: 83,33,99,05,622.01
Calculated sum of cleaned MP amounts: ₹83,339,905,622.01
Integrity Check - Cleaned Sum Matches Official Grand Total: True


## 6. Systematic Column Role Identification
Classify every column in the dataset into analytical domains:
- **Financial / Amount Columns**
- **Geographic / Location Columns**
- **MP / Political Representative Columns**
- **Project / Work Identifiers**
- **Date / Timeline Columns**
- **Progress / Status Columns**
- **Vendor / Payment Columns**


In [7]:
column_roles = {
    "Sr. No.": {
        "Type": "Categorical / Ordinal",
        "Role": "Serial Number Index",
        "Domain": "Identifier (Row-level only)",
        "Usefulness": "Data sequencing only; not a domain entity ID."
    },
    "State": {
        "Type": "Categorical (Nominal)",
        "Role": "State / Union Territory",
        "Domain": "Geographic Analysis",
        "Usefulness": "State-level aggregation, regional allocation disparities, geographic dashboard."
    },
    "Hon'ble Members of Parliaments": {
        "Type": "Categorical (Text)",
        "Role": "Member of Parliament (Lok Sabha)",
        "Domain": "Representative Profile",
        "Usefulness": "MP profile tracking, allocation audit per representative."
    },
    "Constituency": {
        "Type": "Categorical (Text)",
        "Role": "Lok Sabha Parliamentary Constituency",
        "Domain": "Geographic / Administrative Entity",
        "Usefulness": "Constituency-level budget analysis, GIS boundary mapping."
    },
    "Allocated AMOUNT ( ₹ )": {
        "Type": "Financial (Numeric INR)",
        "Role": "Total Allocated MPLADS Fund Limit per MP",
        "Domain": "Financial Risk Intelligence",
        "Usefulness": "Baseline allocation limit, allocation outlier detection, state budget variance."
    }
}

roles_df = pd.DataFrame.from_dict(column_roles, orient='index')
print(roles_df.to_string())


                                                   Type                                      Role                              Domain                                                                       Usefulness
Sr. No.                           Categorical / Ordinal                       Serial Number Index         Identifier (Row-level only)                                    Data sequencing only; not a domain entity ID.
State                             Categorical (Nominal)                   State / Union Territory                 Geographic Analysis  State-level aggregation, regional allocation disparities, geographic dashboard.
Hon'ble Members of Parliaments       Categorical (Text)          Member of Parliament (Lok Sabha)              Representative Profile                        MP profile tracking, allocation audit per representative.
Constituency                         Categorical (Text)      Lok Sabha Parliamentary Constituency  Geographic / Administrative Entity       

## 7. Statistical Analysis & Financial Distribution
Analyze the distribution of MPLADS fund limits across Lok Sabha MPs.


In [8]:
amt = df_cleaned['allocated_amount_clean']
stats = {
    "Total Lok Sabha MPs in Data": len(df_cleaned),
    "Total Fund Allocated (INR)": f"₹{amt.sum():,.2f}",
    "Minimum Allocation": f"₹{amt.min():,.2f}",
    "Maximum Allocation": f"₹{amt.max():,.2f}",
    "Mean Allocation": f"₹{amt.mean():,.2f}",
    "Median Allocation": f"₹{amt.median():,.2f}",
    "Standard Deviation": f"₹{amt.std():,.2f}",
    "25th Percentile (Q1)": f"₹{amt.quantile(0.25):,.2f}",
    "75th Percentile (Q3)": f"₹{amt.quantile(0.75):,.2f}",
    "IQR (Interquartile Range)": f"₹{(amt.quantile(0.75) - amt.quantile(0.25)):,.2f}"
}

for k, v in stats.items():
    print(f"{k:35s}: {v}")

print("\n--- Top 5 Most Common Allocation Amounts ---")
freq_df = amt.value_counts().head(5).reset_index()
freq_df.columns = ['Allocated Amount (INR)', 'Number of MPs']
freq_df['Percentage of MPs (%)'] = (freq_df['Number of MPs'] / len(df_cleaned)) * 100
freq_df['Formatted Amount'] = freq_df['Allocated Amount (INR)'].apply(lambda x: f"₹{x:,.2f}")
print(freq_df[['Formatted Amount', 'Number of MPs', 'Percentage of MPs (%)']].to_string(index=False))


Total Lok Sabha MPs in Data        : 543
Total Fund Allocated (INR)         : ₹83,339,905,622.01
Minimum Allocation                 : ₹49,000,000.00
Maximum Allocation                 : ₹327,477,390.86
Mean Allocation                    : ₹153,763,663.51
Median Allocation                  : ₹147,000,000.00
Standard Deviation                 : ₹20,955,462.28
25th Percentile (Q1)               : ₹147,000,000.00
75th Percentile (Q3)               : ₹148,413,830.11
IQR (Interquartile Range)          : ₹1,413,830.11

--- Top 5 Most Common Allocation Amounts ---
Formatted Amount  Number of MPs  Percentage of MPs (%)
 ₹147,000,000.00            382                  70.35
 ₹171,500,000.00              4                   0.74
 ₹269,500,000.00              3                   0.55
  ₹98,000,000.00              2                   0.37
 ₹190,289,442.00              1                   0.18


## 8. Outlier & Disparity Identification (Risk Signal Analysis)
Identify MPs with allocations significantly differing from the standard modal allocation (₹14.70 Crore).
*Note: In accordance with our core principle, these are flagged as statistical risk signals/deviations for administrative verification, NOT fraud.*


In [9]:
# Top 5 Highest Allocations
print("=== Top 5 Highest MP Allocations ===")
top_5 = df_cleaned.sort_values(by='allocated_amount_clean', ascending=False).head(5)
print(top_5[['State', 'Hon\'ble Members of Parliaments', 'Constituency', 'Allocated AMOUNT ( ₹ )']].to_string(index=False))

# Top 5 Lowest Allocations
print("\n=== Top 5 Lowest MP Allocations ===")
bottom_5 = df_cleaned.sort_values(by='allocated_amount_clean', ascending=True).head(5)
print(bottom_5[['State', 'Hon\'ble Members of Parliaments', 'Constituency', 'Allocated AMOUNT ( ₹ )']].to_string(index=False))


=== Top 5 Highest MP Allocations ===
      State Hon'ble Members of Parliaments Constituency Allocated AMOUNT ( ₹ )
  Telangana                EATALA RAJENDER   MALKAJGIRI           327477390.86
  Telangana              Arvind Dharmapuri    NIZAMABAD           281396355.11
West Bengal                 Asit Kumar Mal   BOLPUR(SC)           275694456.74
  Karnataka    Ramesh Chandappa Jigajinagi  BIJAPUR(SC)              269500000
     Odisha         Saptagiri Sankar Ulaka  KORAPUT(ST)              269500000

=== Top 5 Lowest MP Allocations ===
      State Hon'ble Members of Parliaments Constituency Allocated AMOUNT ( ₹ )
West Bengal                 SK NURUL ISLAM     BASIRHAT               49000000
      Assam               Pradyut Bordoloi      NOWGONG               98000000
  Meghalaya              ANDREW J. SYNGKON     SHILLONG               98000000
     Kerala          Priyanka Gandhi Vadra      WAYANAD              122500000
Maharashtra    BHAUSAHEB RAJARAM WAKCHAURE   SHIRDI(SC)  

## 9. State-wise Aggregation & Distribution
Aggregate allocation totals and MP counts by State/UT.


In [10]:
state_summary = df_cleaned.groupby('State').agg(
    total_allocated=('allocated_amount_clean', 'sum'),
    mp_count=('allocated_amount_clean', 'count'),
    mean_allocation=('allocated_amount_clean', 'mean')
).sort_values(by='total_allocated', ascending=False)

state_summary['total_allocated_cr'] = state_summary['total_allocated'] / 1e7
state_summary['mean_allocation_cr'] = state_summary['mean_allocation'] / 1e7

print("=== Top 10 States by Total MPLADS Allocation ===")
print(state_summary[['mp_count', 'total_allocated_cr', 'mean_allocation_cr']].head(10).to_string())


=== Top 10 States by Total MPLADS Allocation ===
                mp_count  total_allocated_cr  mean_allocation_cr
State                                                           
Uttar Pradesh         80             1211.18               15.14
Maharashtra           48              734.23               15.30
West Bengal           42              639.13               15.22
Tamil Nadu            39              615.92               15.79
Bihar                 40              599.95               15.00
Madhya Pradesh        29              443.05               15.28
Karnataka             28              427.53               15.27
Andhra Pradesh        25              406.07               16.24
Gujarat               26              384.70               14.80
Rajasthan             25              376.76               15.07


## 10. Summary of Analytical Readiness & Schema Gaps

| Capability Domain | Supported in Current Dataset? | Reason / Data Requirement |
| :--- | :---: | :--- |
| **MP Fund Allocation Profiling** | **YES** | Available via `Allocated AMOUNT ( ₹ )`, `State`, and `Constituency`. |
| **Macro Financial Outlier Signals** | **YES** | Outliers from ₹14.70 Cr baseline can be flagged using Isolation Forest / Z-score. |
| **Geographic State/Constituency Distribution** | **YES** | 36 States/UTs and 542 Lok Sabha constituencies present. |
| **Work-level Delay & Timeline Analysis** | **NO** | Requires work sanction date, start date, and completion date. |
| **Payment vs. Progress Consistency** | **NO** | Requires installment payments, physical progress (%), and utilization certificates. |
| **Duplicate / Highly Similar Work NLP** | **NO** | Requires project titles, work descriptions, and sanction categories. |
| **Implementing Agency & Vendor Intelligence** | **NO** | Requires executing agency names, contractor IDs, and invoice details. |

### Next Step:
Proceed to build Phase 1 Data Cleaning & Validation pipeline, and ingest granular project-level eSAKSHI data exports as they become available.
